# 07: Fixes
In v1 the termination rate was calculated over every registered trial, including ones that are still running and haven't had the chance to stop yet. This notebook redoes it only on trials that actually ended, replaces the hardcoded average and checks if the difference between phases is real or just chance.

In [69]:
import pandas as pd
import sys
sys.path.append("..")
from src.fixed_data import fix_data
from src.analysis import termination_by_phase
from scipy.stats import chi2_contingency
df = pd.read_csv("../data/processed/alzheimer.csv")
df = fix_data(df)

## Which trials count as finished

I kept only COMPLETED and TERMINATED, because those are the only two where the trial actually ran and reached an end.

What I left out and why:
- RECRUITING, NOT_YET_RECRUITING, ACTIVE_NOT_RECRUITING, ENROLLING_BY_INVITATION: still running
- SUSPENDED: paused, it can restart
- WITHDRAWN: stopped before anyone enrolled, so it never really ran
- UNKNOWN: the sponsor stopped updating the record, so we don't know how it ended
- AVAILABLE, NO_LONGER_AVAILABLE: expanded access programmes, not trials

In [70]:
finished =df[df["status"].isin(["TERMINATED","COMPLETED"])]

## Termination rate by phase: all registered vs finished only

Every rate goes up when I only count finished trials, which makes sense since the trials that can't have stopped yet are gone. Phase 3 is still the worst in both: 20.0% of all registered Phase 3 trials and 26.8% of finished ones.

In [ ]:
phase_all =termination_by_phase(df)

PHASE3            20.00
PHASE2, PHASE3    16.67
EARLY_PHASE1      12.12
PHASE2            12.11
PHASE1             9.21
PHASE4             5.71
PHASE1, PHASE2     4.81
dtype: float64

In [ ]:
phase_finished =termination_by_phase(finished)

PHASE3            26.79
PHASE2, PHASE3    23.81
EARLY_PHASE1      17.39
PHASE2            17.03
PHASE1            11.31
PHASE1, PHASE2     9.09
PHASE4             8.25
dtype: float64

In [ ]:
pd.DataFrame({"all registered":phase_all,"finished only":phase_finished})

,all registered,finished only
EARLY_PHASE1,12.12,17.39
PHASE1,9.21,11.31
"PHASE1, PHASE2",4.81,9.09
PHASE2,12.11,17.03
"PHASE2, PHASE3",16.67,23.81
PHASE3,20.00,26.79
PHASE4,5.71,8.25


## Overall termination rate

In v1 I hardcoded 7.43. Now it's calculated from the data: 7.41% over all registered trials and 12.2% over finished ones. The small change from 7.43 is because the data was downloaded again and ClinicalTrials.gov had added 8 trials since (4,092 to 4,100).

In [78]:
overall_rate =(((df["status"] == "TERMINATED").mean()) * 100).round(2)

In [81]:
finished_rate =(((finished["status"] == "TERMINATED").mean()) * 100).round(2)

In [82]:
print(overall_rate,finished_rate)

7.41 12.2


## Is the difference between phases real?

I used a chi-square test on phase vs terminated or completed, on finished trials that have a phase (about 1,260, since observational studies have no phase).

p ≈ 0.000007, so termination rate really does depend on phase and it's not chance. The smallest expected count is 6.8, so the test is valid. This doesn't prove Phase 3 alone is significantly higher than every other phase, only that the phases differ overall.

In [76]:
table_terminated = pd.crosstab(finished["phases"],finished["status"])

In [77]:
chi2,p,dof,expected = chi2_contingency(table_terminated)
print(p)
print(expected)

6.962941312655235e-06
[[ 38.53955696   7.46044304]
 [325.91060127  63.08939873]
 [ 46.07990506   8.92009494]
 [344.34256329  66.65743671]
 [ 35.18829114   6.81170886]
 [187.67088608  36.32911392]
 [ 81.2681962   15.7318038 ]]
